# Notebook 04 — SPI Drought Events × Multi-Source News Corroboration

**Goal**: For each SPI drought event, find news from global sources (not just Google) in multiple languages to confirm real-world impact.

**Pipeline**:
```
SPI events (CSV)
  → filter severe events
  → GDELT + ReliefWeb + GDACS + Google News (multi-source)
  → langdetect → XLM-R pre-filter → Helsinki-NLP translate → ClimateBERT
  → LLM assess → corroboration score
```

**Why multi-source?** Google News misses regional African/Asian press. GDELT indexes 65,000+ sources in 100+ languages.

**Why multilingual?** Drought news in Kenya appears in Swahili. In Pakistan in Urdu. In Senegal in French. English-only pipelines miss 60%+ of coverage.

## Step 1 — Install Dependencies

In [ ]:
import sys, subprocess
pkgs = [
    'feedparser', 'transformers', 'torch', 'huggingface_hub',
    'trafilatura', 'langdetect', 'requests',
    'matplotlib', 'seaborn', 'pandas', 'numpy',
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], capture_output=True)
print("All dependencies installed.")

## Step 2 — Imports & Config

In [ ]:
import os, ssl, json, re, urllib.request, warnings, time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import feedparser
import trafilatura
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────
SPI_CSV = os.path.expanduser(
    "~/Library/CloudStorage/Box-Box/R - Projects - Tharaka/"
    "Drought-indices/Data/drought_summary_yearly_bycountry.csv"
)
HF_TOKEN = os.environ.get('HF_TOKEN', '')

# ── Thresholds ───────────────────────────────────────────────────────────
MIN_PCT_PIXELS   = 30   # ≥30% of country pixels in drought
TOP_N_EVENTS     = 20   # process top-N worst events
MAX_ARTICLES_SRC = 10   # articles per source per event

# ── News sources (no registration needed) ────────────────────────────────
GDACS_DROUGHT_RSS = 'https://www.gdacs.org/rss.aspx?profile=ARCHIVE&type=DR'
ALLAFRICA_RSS     = 'https://allafrica.com/drought/rss.xml'

print(f"SPI file: {os.path.exists(SPI_CSV)}")
print(f"HF token: {bool(HF_TOKEN)}")

## Step 3 — Load & Filter SPI Events

In [ ]:
try:
    df_spi = pd.read_csv(SPI_CSV)
    print(f"Loaded {len(df_spi):,} rows | columns: {list(df_spi.columns)}")
except FileNotFoundError:
    print("SPI file not found — using embedded sample events")
    df_spi = pd.DataFrame([
        {'country':'Ethiopia','event_year':2022,'pct_pixels_affected':81,
         'mean_peak_spi':-2.1,'mean_n_severe':1.8,'mean_n_extreme':0.9,
         'mean_total_severity':-11.2,'config_name':'SPI3_cum_thr00_dur1'},
        {'country':'Kenya','event_year':2022,'pct_pixels_affected':72,
         'mean_peak_spi':-1.8,'mean_n_severe':1.2,'mean_n_extreme':0.4,
         'mean_total_severity':-8.1,'config_name':'SPI3_cum_thr00_dur1'},
        {'country':'Somalia','event_year':2022,'pct_pixels_affected':65,
         'mean_peak_spi':-1.9,'mean_n_severe':1.5,'mean_n_extreme':0.6,
         'mean_total_severity':-9.5,'config_name':'SPI3_cum_thr00_dur1'},
        {'country':'Niger','event_year':2021,'pct_pixels_affected':58,
         'mean_peak_spi':-1.6,'mean_n_severe':0.8,'mean_n_extreme':0.1,
         'mean_total_severity':-6.3,'config_name':'SPI3_cum_thr00_dur1'},
        {'country':'Pakistan','event_year':2021,'pct_pixels_affected':52,
         'mean_peak_spi':-1.5,'mean_n_severe':0.7,'mean_n_extreme':0.2,
         'mean_total_severity':-5.8,'config_name':'SPI3_cum_thr00_dur1'},
    ])

# Filter: SPI3, spatial extent, severity
df_f = df_spi.copy()
if 'config_name' in df_f.columns:
    df_f = df_f[df_f['config_name'].str.contains('SPI3', na=False)]
if 'pct_pixels_affected' in df_f.columns:
    df_f = df_f[df_f['pct_pixels_affected'] >= MIN_PCT_PIXELS]
if 'mean_n_severe' in df_f.columns:
    df_f = df_f[(df_f['mean_n_severe'] + df_f.get('mean_n_extreme', 0)) > 0]

df_f = df_f.sort_values('mean_peak_spi', ascending=True)
df_events = df_f.drop_duplicates(['country','event_year'], keep='first').reset_index(drop=True)

print(f"Significant events: {len(df_events)}")
print(f"Years: {df_events['event_year'].min():.0f} – {df_events['event_year'].max():.0f}")
print(df_events[['country','event_year','pct_pixels_affected','mean_peak_spi']].head(10).to_string(index=False))

## Step 4 — Visualise SPI Event Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

yr = df_events['event_year'].value_counts().sort_index()
axes[0].bar(yr.index, yr.values, color='steelblue', edgecolor='white')
axes[0].set_title('SPI Drought Events by Year'); axes[0].set_xlabel('Year')

ct = df_events['country'].value_counts().head(15)
axes[1].barh(ct.index[::-1], ct.values[::-1], color='firebrick', edgecolor='white')
axes[1].set_title('Countries: Most Drought Years'); axes[1].set_xlabel('Years with severe drought')

plt.tight_layout(); plt.savefig('spi_event_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 5 — Multi-Source News Scraper

### Why not just Google News?
| Source | Coverage | Limitation |
|--------|----------|------------|
| Google News RSS | English-dominant, global major press | Misses regional African/Asian outlets |
| **GDELT DOC 2.0** | 65,000+ sources, 100+ languages | Max 250/query, no official SLA |
| **ReliefWeb API** | UN/NGO humanitarian reports, 100+ countries | Formal documents, not breaking news |
| **GDACS RSS** | Disaster alerts incl. drought | Low volume, official only |
| **AllAfrica RSS** | African regional press | Drought-specific feed |

**Strategy**: query all sources, merge, deduplicate by URL.

In [ ]:
# ── SSL context (macOS cert fix) ────────────────────────────────────────
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

HEADERS = {'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                           'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36')}

# ── Source 1: GDELT DOC 2.0 ─────────────────────────────────────────────
def gdelt_doc2(query, year, max_records=50):
    """GDELT DOC 2.0 — free, no auth, 65K+ global sources."""
    start = f"{year}0101000000"
    end   = f"{year}1231235959"
    params = {
        'query': query, 'mode': 'artlist',
        'maxrecords': max_records,
        'startdatetime': start, 'enddatetime': end,
        'format': 'json',
    }
    try:
        r = requests.get(
            'https://api.gdeltproject.org/api/v2/doc/doc',
            params=params, timeout=20
        )
        data = r.json().get('articles', [])
        return [{
            'title':     a.get('title', ''),
            'summary':   a.get('seendate', ''),
            'url':       a.get('url', ''),
            'source':    a.get('domain', 'GDELT'),
            'language':  a.get('language', 'English'),
            'published': a.get('seendate', ''),
            '_src':      'gdelt',
        } for a in data]
    except Exception as e:
        print(f"    GDELT error: {e}")
        return []

# ── Source 2: ReliefWeb API ──────────────────────────────────────────────
# Country → ISO3 mapping (common drought-prone countries)
COUNTRY_ISO3 = {
    'Ethiopia':'ETH','Kenya':'KEN','Somalia':'SOM','Niger':'NER',
    'Mali':'MLI','Sudan':'SDN','Chad':'TCD','Zimbabwe':'ZWE',
    'Mozambique':'MOZ','Madagascar':'MDG','Malawi':'MWI',
    'Pakistan':'PAK','India':'IND','Afghanistan':'AFG',
    'Iraq':'IRQ','Syria':'SYR','Yemen':'YEM','Iran':'IRN',
    'Brazil':'BRA','Argentina':'ARG','Chile':'CHL',
    'Australia':'AUS','United States':'USA','Mexico':'MEX',
}

def reliefweb_drought(country, year, limit=20):
    """ReliefWeb — humanitarian reports filtered by disaster_type=Drought."""
    iso3 = COUNTRY_ISO3.get(country)
    filters = {
        'operator': 'AND',
        'conditions': [
            {'field': 'disaster_type.name', 'value': 'Drought'},
            {'field': 'date.created', 'value': {
                'from': f'{year}-01-01T00:00:00+00:00',
                'to':   f'{year}-12-31T23:59:59+00:00',
            }},
        ]
    }
    if iso3:
        filters['conditions'].append({'field': 'country.iso3', 'value': iso3})
    payload = {
        'filter': filters,
        'fields': {'include': ['title','date','source','url','language','body']},
        'sort': ['date.created:desc'],
        'limit': limit,
    }
    try:
        r = requests.post('https://api.reliefweb.int/v1/reports',
                          json=payload, timeout=20)
        items = r.json().get('data', [])
        return [{
            'title':     i['fields'].get('title',''),
            'summary':   (i['fields'].get('body','') or '')[:300],
            'url':       i['fields'].get('url','') or f"rw-{i.get('id','')}",
            'source':    'ReliefWeb',
            'language':  (i['fields'].get('language') or [{}])[0].get('name','English'),
            'published': i['fields'].get('date',{}).get('created',''),
            '_src':      'reliefweb',
        } for i in items]
    except Exception as e:
        print(f"    ReliefWeb error: {e}")
        return []

# ── Source 3: Google News RSS (fallback) ─────────────────────────────────
def google_news_rss(query, max_articles=10):
    url = ('https://news.google.com/rss/search?q='
           + query.replace(' ','+') + '&hl=en-US&gl=US&ceid=US:en')
    try:
        req = urllib.request.Request(url, headers=HEADERS)
        with urllib.request.urlopen(req, timeout=15, context=ssl_ctx) as resp:
            raw = resp.read()
        feed = feedparser.parse(raw)
        return [{
            'title':     e.get('title',''),
            'summary':   e.get('summary',''),
            'url':       e.get('link',''),
            'source':    e.get('source',{}).get('title','Google News'),
            'language':  'English',
            'published': e.get('published',''),
            '_src':      'google',
        } for e in feed.entries[:max_articles]]
    except Exception as e:
        print(f"    Google News error: {e}")
        return []

# ── Source 4: RSS feeds (GDACS + AllAfrica) ───────────────────────────────
def rss_feeds_drought(country):
    feeds = [
        ('GDACS',     GDACS_DROUGHT_RSS),
        ('AllAfrica', ALLAFRICA_RSS),
    ]
    articles = []
    for name, url in feeds:
        try:
            feed = feedparser.parse(url)
            for e in feed.entries[:20]:
                title = e.get('title','')
                if country.lower() in title.lower() or not country:
                    articles.append({
                        'title':     title,
                        'summary':   e.get('summary',''),
                        'url':       e.get('link',''),
                        'source':    name,
                        'language':  'English',
                        'published': e.get('published',''),
                        '_src':      'rss',
                    })
        except Exception:
            pass
    return articles

# ── Merge + deduplicate ───────────────────────────────────────────────────
def fetch_all_sources(country, year):
    query = f'drought {country} {year}'
    print(f"  Querying all sources for: {query}")
    all_arts = []
    all_arts += gdelt_doc2(query, year, max_records=MAX_ARTICLES_SRC)
    all_arts += reliefweb_drought(country, year, limit=MAX_ARTICLES_SRC)
    all_arts += google_news_rss(query, max_articles=MAX_ARTICLES_SRC)
    all_arts += rss_feeds_drought(country)
    df_a = pd.DataFrame(all_arts)
    if df_a.empty:
        return df_a
    df_a = df_a[df_a['url'] != ''].drop_duplicates('url').reset_index(drop=True)
    print(f"  Total unique articles: {len(df_a)} "
          f"(GDELT:{(df_a._src=='gdelt').sum()} "
          f"ReliefWeb:{(df_a._src=='reliefweb').sum()} "
          f"Google:{(df_a._src=='google').sum()} "
          f"RSS:{(df_a._src=='rss').sum()})")
    return df_a

# Quick test
test_df = fetch_all_sources('Ethiopia', 2022)
print(f"Test: {len(test_df)} articles")

## Step 6 — Multilingual Processing

### Concept: Why translation matters
A drought in Kenya generates news in **English AND Swahili**. Pakistan drought news appears in **Urdu**. West Africa in **French**. English-only pipeline misses 50–70% of coverage.

### Two-stage approach
```
article text
    │
    ├── langdetect → language code (fr, ar, sw, hi...)
    │
    ├── Stage 1: XLM-RoBERTa NLI → drought probability score
    │          (multilingual, no translation needed)
    │          score < 0.35 → discard (saves translation cost)
    │
    └── Stage 2 (only ~20-30% of articles):
               Helsinki-NLP → translate to English
               ClimateBERT → domain filter
```

| Model | HuggingFace ID | Size | Role |
|-------|---------------|------|------|
| XLM-R NLI | `joeddav/xlm-roberta-large-xnli` | ~1.1GB | Stage 1 multilingual pre-filter |
| Helsinki (per lang) | `Helsinki-NLP/opus-mt-{src}-en` | ~300MB each | Translate to English |
| ClimateBERT | `climatebert/distilroberta-base-climate-f` | ~330MB | Climate domain filter |

In [ ]:
from transformers import pipeline as hf_pipeline

# ── Language detection ────────────────────────────────────────────────────
try:
    from langdetect import detect as detect_lang, LangDetectException
    LANGDETECT_OK = True
    print("langdetect: OK")
except ImportError:
    LANGDETECT_OK = False
    print("langdetect not installed — defaulting all articles to English")

def get_lang(text):
    if not LANGDETECT_OK or not text:
        return 'en'
    try:
        return detect_lang(text[:200])
    except:
        return 'en'

# ── Stage 1: XLM-R multilingual pre-filter ───────────────────────────────
XLM_LABELS = ['drought disaster', 'unrelated topic']

print("Loading XLM-RoBERTa NLI (multilingual pre-filter)...")
try:
    xlmr = hf_pipeline(
        'zero-shot-classification',
        model='joeddav/xlm-roberta-large-xnli',
        device=-1
    )
    XLM_OK = True
    print("XLM-R loaded.")
except Exception as e:
    print(f"XLM-R failed: {e}")
    XLM_OK = False

def xlmr_drought_score(text):
    """Returns probability 0-1 that text is about drought (works in any language)."""
    if not XLM_OK:
        return 0.5
    try:
        r = xlmr(text[:512], candidate_labels=XLM_LABELS, multi_label=False)
        idx = r['labels'].index('drought disaster')
        return r['scores'][idx]
    except:
        return 0.0

# ── Stage 2a: Helsinki-NLP translators (lazy-loaded) ─────────────────────
HELSINKI_MAP = {
    'fr': 'Helsinki-NLP/opus-mt-fr-en',
    'ar': 'Helsinki-NLP/opus-mt-ar-en',
    'sw': 'Helsinki-NLP/opus-mt-swc-en',
    'am': 'Helsinki-NLP/opus-mt-am-en',
    'hi': 'Helsinki-NLP/opus-mt-hi-en',
    'pt': 'Helsinki-NLP/opus-mt-ROMANCE-en',
    'es': 'Helsinki-NLP/opus-mt-ROMANCE-en',
    'de': 'Helsinki-NLP/opus-mt-de-en',
    'zh': 'Helsinki-NLP/opus-mt-zh-en',
}
_translators = {}

def translate_to_english(text, lang):
    """Translate text to English using Helsinki-NLP. Cache loaded models."""
    if lang == 'en' or lang not in HELSINKI_MAP:
        return text
    if lang not in _translators:
        try:
            print(f"    Loading translator: {HELSINKI_MAP[lang]}")
            _translators[lang] = hf_pipeline(
                'translation',
                model=HELSINKI_MAP[lang],
                device=-1
            )
        except Exception as e:
            print(f"    Translator load failed ({lang}): {e}")
            return text
    try:
        result = _translators[lang](text[:512], max_length=512)
        return result[0]['translation_text']
    except:
        return text

# ── Stage 2b: ClimateBERT ─────────────────────────────────────────────────
print("Loading ClimateBERT...")
try:
    climatebert = hf_pipeline(
        'text-classification',
        model='climatebert/distilroberta-base-climate-f',
        device=-1, truncation=True, max_length=128
    )
    CLIMATEBERT_OK = True
    print("ClimateBERT loaded.")
except Exception as e:
    print(f"ClimateBERT failed: {e}")
    CLIMATEBERT_OK = False

DROUGHT_KW = ['drought','dry spell','water shortage','rainfall deficit',
              'crop failure','famine','water scarcity','precipitation deficit']

def process_article(title, summary, source_language='en'):
    """
    Full multilingual processing pipeline.
    Returns: {'passes': bool, 'xlmr_score': float, 'lang': str, 'translated': bool}
    """
    text = (title + ' ' + summary)[:600]
    lang = source_language if source_language != 'English' else 'en'
    if lang == 'English': lang = 'en'

    # Detect language from text if not provided
    if lang == 'en':
        detected = get_lang(text)
        if detected != 'en':
            lang = detected

    # Stage 1: fast multilingual pre-filter
    xlmr_score = xlmr_drought_score(text)
    if xlmr_score < 0.35:
        return {'passes': False, 'xlmr_score': xlmr_score, 'lang': lang, 'translated': False}

    # Stage 2a: translate if non-English
    en_text = text
    translated = False
    if lang != 'en' and lang in HELSINKI_MAP:
        en_text = translate_to_english(text, lang)
        translated = True

    # Stage 2b: ClimateBERT on English text
    if CLIMATEBERT_OK:
        try:
            result = climatebert(en_text[:512])[0]
            passes = result['label'].lower() in ('yes', 'climate', 'label_1')
        except:
            passes = any(kw in en_text.lower() for kw in DROUGHT_KW)
    else:
        passes = any(kw in en_text.lower() for kw in DROUGHT_KW)

    return {
        'passes': passes,
        'xlmr_score': round(xlmr_score, 3),
        'lang': lang,
        'translated': translated,
        'en_text': en_text[:400],
    }

print("Multilingual pipeline ready.")
print(f"Supported translation langs: {list(HELSINKI_MAP.keys())}")

## Step 7 — LLM Drought Assessment (optional — needs HF_TOKEN)

Works on **English text** (after translation in Step 6).
Keyword fallback used if no token.

In [ ]:
LLM_OK = False
if HF_TOKEN:
    try:
        from huggingface_hub import InferenceClient
        llm = InferenceClient(model='mistralai/Mistral-7B-Instruct-v0.2', token=HF_TOKEN)
        LLM_OK = True
        print("LLM ready: Mistral-7B")
    except Exception as e:
        print(f"LLM init failed: {e}")
else:
    print("No HF_TOKEN — keyword fallback active.")

FEW_SHOT = """You are a drought impact classifier. Respond with valid JSON only.

Example: "Severe drought grips Horn of Africa, millions face starvation"
Output: {"is_drought": true, "location": "Horn of Africa", "severity": "severe", "population_impact": "millions"}

Example: "Stock markets rally as tech earnings beat"
Output: {"is_drought": false, "location": null, "severity": null, "population_impact": null}
"""

SEV_MAP = {'mild':1,'moderate':2,'severe':3,'extreme':4}

def assess_article(en_text):
    text = en_text[:400]
    if LLM_OK:
        prompt = f'<s>[INST] {FEW_SHOT}\nClassify: "{text}" [/INST]'
        try:
            raw = llm.text_generation(prompt, max_new_tokens=150, temperature=0.1)
            m = re.search(r'\{.*?\}', raw, re.DOTALL)
            if m:
                return json.loads(m.group())
        except:
            pass
    # Keyword fallback
    tl = text.lower()
    is_drought = any(kw in tl for kw in DROUGHT_KW)
    sev = next((s for s in ['extreme','severe','moderate','mild'] if s in tl), None)
    pop = next((t for t in ['million','thousand','people','population','household'] if t in tl), None)
    return {'is_drought': is_drought, 'severity': sev, 'location': None,
            'population_impact': pop, 'method': 'keyword'}

print("Assessment function ready.")

## Step 8 — Run SPI × News Correlation Pipeline

For each of the top-20 worst SPI events:
1. Fetch from GDELT + ReliefWeb + Google News + RSS
2. Multilingual processing (XLM-R → translate → ClimateBERT)
3. LLM assess each filtered article
4. Compute corroboration score

**Corroboration score** = articles confirming drought / total articles found
- ≥0.7 → CONFIRMED (SPI + news agree)
- 0.3–0.7 → PARTIAL
- <0.3 → NO EVIDENCE

In [ ]:
events_to_run = df_events.head(TOP_N_EVENTS).copy()
results = []
lang_stats = {}  # track which languages we saw

for idx, row in events_to_run.iterrows():
    country   = row['country']
    year      = int(row['event_year'])
    peak_spi  = float(row.get('mean_peak_spi', 0))
    pct       = float(row.get('pct_pixels_affected', 0))

    print(f"\n[{idx+1}/{len(events_to_run)}] {country} {year}  "
          f"SPI={peak_spi:.2f}  {pct:.0f}% pixels")

    # 1. Fetch from all sources
    df_arts = fetch_all_sources(country, year)
    n_found = len(df_arts)

    if df_arts.empty:
        results.append({'country':country,'event_year':year,'peak_spi':peak_spi,
                        'pct_pixels_affected':pct,'n_found':0,'n_filtered':0,
                        'n_confirmed':0,'corroboration_score':0.0,
                        'langs_seen':'','population_impact':False,'avg_severity':None})
        continue

    # 2. Multilingual processing + LLM assess
    n_filtered = 0
    n_confirmed = 0
    severities = []
    pop_impact = False
    langs_seen = set()

    for _, art in df_arts.iterrows():
        proc = process_article(
            art.get('title',''),
            art.get('summary',''),
            art.get('language','English')
        )
        langs_seen.add(proc['lang'])
        lang_stats[proc['lang']] = lang_stats.get(proc['lang'], 0) + 1

        if not proc['passes']:
            continue
        n_filtered += 1

        en_text = proc.get('en_text', art.get('title','') + ' ' + art.get('summary',''))
        assessment = assess_article(en_text)

        if assessment.get('is_drought'):
            n_confirmed += 1
            if assessment.get('severity'):
                severities.append(assessment['severity'])
            if assessment.get('population_impact'):
                pop_impact = True

    score = n_confirmed / n_found if n_found > 0 else 0.0

    avg_sev = None
    if severities:
        nums = [SEV_MAP.get(s, 0) for s in severities]
        n = round(np.mean(nums))
        avg_sev = {1:'mild',2:'moderate',3:'severe',4:'extreme'}.get(n,'severe')

    print(f"  Found:{n_found}  Filtered:{n_filtered}  Confirmed:{n_confirmed}  "
          f"Score:{score:.2f}  Langs:{langs_seen}")

    results.append({
        'country': country, 'event_year': year,
        'peak_spi': peak_spi, 'pct_pixels_affected': pct,
        'n_found': n_found, 'n_filtered': n_filtered, 'n_confirmed': n_confirmed,
        'corroboration_score': round(score, 3),
        'langs_seen': ','.join(sorted(langs_seen)),
        'population_impact': pop_impact,
        'avg_severity': avg_sev,
    })
    time.sleep(0.5)

df_res = pd.DataFrame(results)
print("\n=== Pipeline complete ===")
print(df_res[['country','event_year','n_found','n_confirmed','corroboration_score','langs_seen']].to_string(index=False))

## Step 9 — Language Distribution

See which languages appeared in the news — confirms value of multilingual pipeline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: language distribution across all articles
if lang_stats:
    langs_df = pd.Series(lang_stats).sort_values(ascending=False).head(12)
    colors = ['#2196F3' if l == 'en' else '#FF9800' for l in langs_df.index]
    axes[0].bar(langs_df.index, langs_df.values, color=colors, edgecolor='white')
    axes[0].set_title('Article Languages Detected')
    axes[0].set_xlabel('Language code')
    axes[0].set_ylabel('Article count')
    non_en = sum(v for k,v in lang_stats.items() if k != 'en')
    total  = sum(lang_stats.values())
    axes[0].set_title(f'Article Languages ({non_en}/{total} = {non_en/total*100:.0f}% non-English)')

# Right: corroboration score by country
df_plot = df_res.sort_values('corroboration_score', ascending=False)
labels  = [f"{r['country']} {int(r['event_year'])}" for _, r in df_plot.iterrows()]
colors2 = ['#d73027' if s>=0.7 else '#fee090' if s>=0.3 else '#91bfdb'
           for s in df_plot['corroboration_score']]
axes[1].barh(labels[::-1], df_plot['corroboration_score'][::-1],
             color=colors2[::-1], edgecolor='white')
axes[1].axvline(0.7, color='red', ls='--', alpha=0.6, label='CONFIRMED ≥0.7')
axes[1].axvline(0.3, color='orange', ls='--', alpha=0.6, label='PARTIAL ≥0.3')
axes[1].set_xlabel('Corroboration Score')
axes[1].set_title('SPI Events Ranked by News Corroboration')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('spi_news_multilingual.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 10 — SPI Intensity vs News Corroboration

Expected: worse SPI (more negative) → higher corroboration score.
Exceptions reveal events that are real but underreported (remote areas) or well-reported minor events.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df_res['peak_spi'], df_res['corroboration_score'],
                c=df_res['corroboration_score'], cmap='RdYlBu_r',
                s=100, edgecolors='gray', alpha=0.85)
axes[0].set_xlabel('Peak SPI (more negative = worse)')
axes[0].set_ylabel('Corroboration Score')
axes[0].set_title('SPI Intensity vs News Corroboration')
for _, r in df_res.iterrows():
    axes[0].annotate(r['country'][:3], (r['peak_spi'], r['corroboration_score']),
                     fontsize=7, alpha=0.7, xytext=(3,3), textcoords='offset points')

axes[1].scatter(df_res['pct_pixels_affected'], df_res['corroboration_score'],
                c=df_res['corroboration_score'], cmap='RdYlBu_r',
                s=100, edgecolors='gray', alpha=0.85)
axes[1].set_xlabel('% Pixels Affected (spatial extent)')
axes[1].set_ylabel('Corroboration Score')
axes[1].set_title('Spatial Extent vs News Corroboration')

plt.tight_layout()
plt.savefig('spi_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 11 — Validated Event Table

In [ ]:
def cat(s):
    if s >= 0.7: return 'CONFIRMED'
    if s >= 0.3: return 'PARTIAL'
    return 'NO_EVIDENCE'

df_res['status'] = df_res['corroboration_score'].apply(cat)

for status in ['CONFIRMED','PARTIAL','NO_EVIDENCE']:
    sub = df_res[df_res['status']==status].sort_values('corroboration_score',ascending=False)
    print(f"\n{'='*55}")
    print(f"{status}: {len(sub)} events")
    if len(sub):
        cols = ['country','event_year','peak_spi','pct_pixels_affected',
                'corroboration_score','avg_severity','population_impact','langs_seen']
        cols = [c for c in cols if c in sub.columns]
        print(sub[cols].to_string(index=False))

## Step 12 — Export

In [ ]:
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
path = f'spi_news_corroboration_{ts}.csv'
df_res.to_csv(path, index=False)
print(f"Saved: {path}")
print(f"\nSummary:")
print(df_res['status'].value_counts().to_string())
print(f"\nMean corroboration: {df_res['corroboration_score'].mean():.3f}")
if lang_stats:
    non_en_pct = 100*sum(v for k,v in lang_stats.items() if k!='en')/sum(lang_stats.values())
    print(f"Non-English articles: {non_en_pct:.1f}%")
    print(f"Languages seen: {sorted(lang_stats.keys())}")

## Step 13 — What We Built

```
SPI drought events (CSV from Box)
         │
    filter: SPI3, ≥30% pixels, Severe/Extreme class
         │
    ┌─────┴──────────────────────────────────┐
    │ GDELT DOC 2.0   │ ReliefWeb │ Google    │ RSS
    │ 65K+ sources    │ Humanitarian│ News RSS │ (GDACS/AllAfrica)
    └─────────────────┴────────────┴──────────┴──
         │
    langdetect → language code
         │
    XLM-RoBERTa NLI → drought score (works in ANY language)
    discard if score < 0.35  [~70% of articles]
         │
    Helsinki-NLP translate → English  [only ~30% reach here]
         │
    ClimateBERT → climate domain filter
         │
    Mistral-7B → {is_drought, severity, population_impact}
         │
    corroboration_score = n_confirmed / n_found
         │
    CONFIRMED / PARTIAL / NO_EVIDENCE
```

### Why each component
| Component | Why |
|-----------|-----|
| GDELT DOC 2.0 | Covers regional press Google misses (AllAfrica, Dawn, The Hindu) |
| ReliefWeb | Humanitarian situation reports = gold standard for crisis coverage |
| XLM-RoBERTa | Filters non-drought articles in any language — no translation cost |
| Helsinki-NLP | Lightweight translators (~300MB each, CPU-friendly) |
| ClimateBERT | Domain-tuned on climate text — fewer false positives than keywords |

### Next steps
- Add pixel-level `all_drought_events_pixelbypixel.csv` for finer location matching
- Add ReliefWeb `body` field for full article text → better LLM extraction
- Fine-tune XLM-RoBERTa on drought-specific multilingual examples
- Add FEWS NET / ACLED as fourth corroboration source